# PowerBid — Turkish Day-Ahead Market Bidding Assistant

AI-powered bidding decision support system for the Turkish electricity day-ahead market (GÖP).

## Architecture
```
Layer 1 — Data      : EPİAŞ API (PTF, generation, consumption) + Open-Meteo (weather)
Layer 2 — Model     : Random Forest price forecasting
Layer 3 — Decision  : PTF forecast + confidence interval + bid recommendation
Output              : Daily bid report (24 hours)
```

## Sections
1. Setup & imports
2. Data collection (EPİAŞ + Open-Meteo)
3. Feature engineering
4. Model training (v1 → v2)
5. Decision engine
6. Daily bid report

## 1 — Setup & Imports

In [1]:
# Install dependencies
!pip install eptr2 -q
print('eptr2 installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 5.7 MB/s eta 0:00:00
eptr2 installed.


In [2]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from eptr2 import EPTR2
from google.colab import userdata
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# EPİAŞ credentials — set in Colab Secrets (key icon in left sidebar)
# Name: EPIAS_USERNAME, EPIAS_PASSWORD
username = userdata.get('EPIAS_USERNAME')
password = userdata.get('EPIAS_PASSWORD')
eptr = EPTR2(username=username, password=password)
print('EPİAŞ client ready.')

Dotenv file not found at .env.
EPİAŞ client ready.


---
## 2 — Data Collection

### 2a — EPİAŞ: PTF (Market Clearing Price)

PTF = Piyasa Takas Fiyatı = Market Clearing Price.  
The hourly day-ahead price in TRY/MWh set by the merit order intersection.  
Data period: April 2025 – April 2026 (8,784 hours).

In [3]:
# Fetch PTF (day-ahead price)
print('Fetching PTF data...')
result = eptr.call('mcp', start_date='2025-04-01', end_date='2026-04-01')

df_tr = result.copy()
df_tr['datetime'] = pd.to_datetime(df_tr['date']).dt.tz_localize(None)
df_tr['hour']     = df_tr['datetime'].dt.hour
df_tr['month']    = df_tr['datetime'].dt.month
df_tr = df_tr.dropna(subset=['price']).reset_index(drop=True)

print(f'PTF dataset    : {len(df_tr)} hours')
print(f'Min price      : {df_tr["price"].min():.2f} TRY/MWh')
print(f'Max price      : {df_tr["price"].max():.2f} TRY/MWh')
print(f'Avg price      : {df_tr["price"].mean():.2f} TRY/MWh')
print(f'Zero-price hrs : {(df_tr["price"] == 0).sum()}')

Fetching PTF data...
PTF dataset    : 8784 hours
Min price      : 0.00 TRY/MWh
Max price      : 3400.00 TRY/MWh
Avg price      : 2569.19 TRY/MWh
Zero-price hrs : 77


### 2b — EPİAŞ: Generation & Consumption

EPİAŞ limits `rt-gen` and `rt-cons` to 3-month windows.  
We use a quarterly loop to fetch the full year.

In [4]:
def fetch_quarterly(call_name, start, end):
    """Fetch data in quarterly chunks to respect EPİAŞ 3-month limit."""
    periods = [
        ('2025-04-01', '2025-06-30'),
        ('2025-07-01', '2025-09-30'),
        ('2025-10-01', '2025-12-31'),
        ('2026-01-01', '2026-04-01'),
    ]
    dfs = []
    for s, e in periods:
        try:
            df = eptr.call(call_name, start_date=s, end_date=e)
            dfs.append(df)
            print(f'  {s} -> {e}: {len(df)} rows')
            time.sleep(1)
        except Exception as ex:
            print(f'  {s} -> {e}: Error — {ex}')
    return pd.concat(dfs, ignore_index=True) if dfs else None

print('Fetching generation data...')
rt_gen = fetch_quarterly('rt-gen', '2025-04-01', '2026-04-01')
rt_gen['datetime']  = pd.to_datetime(rt_gen['date']).dt.tz_localize(None)
rt_gen['renewable'] = rt_gen['wind'] + rt_gen['sun'] + \
                      rt_gen['river'] + rt_gen['geothermal']
rt_gen = rt_gen.drop_duplicates(subset=['datetime']).reset_index(drop=True)
print(f'Generation rows: {len(rt_gen)}')

print('\nFetching consumption data...')
rt_cons_raw = fetch_quarterly('rt-cons', '2025-04-01', '2026-04-01')
rt_cons_raw.columns = ['date', 'consumption_time', 'consumption_mwh', 'datetime']
rt_cons_raw['datetime']         = pd.to_datetime(rt_cons_raw['datetime'])
rt_cons_raw['consumption_mwh']  = pd.to_numeric(
    rt_cons_raw['consumption_mwh'], errors='coerce')
rt_cons = rt_cons_raw[['datetime','consumption_mwh']].drop_duplicates(
    subset=['datetime']).reset_index(drop=True)
print(f'Consumption rows: {len(rt_cons)}')

Fetching generation data...
  2025-04-01 -> 2025-06-30: 2184 rows
  2025-07-01 -> 2025-09-30: 2208 rows
  2025-10-01 -> 2025-12-31: 2208 rows
  2026-01-01 -> 2026-04-01: 2184 rows
Generation rows: 8784

Fetching consumption data...
  2025-04-01 -> 2025-06-30: 2184 rows
  2025-07-01 -> 2025-09-30: 2208 rows
  2025-10-01 -> 2025-12-31: 2208 rows
  2026-01-01 -> 2026-04-01: 2184 rows


ValueError: Length mismatch: Expected axis has 3 elements, new values have 4 elements

### 2c — Open-Meteo: Historical & Forecast Weather

Free weather API — no registration required.  
Location: Ankara (lat=39.9, lon=32.8) as proxy for Turkish grid centre.

In [ ]:
def get_weather_historical(lat=39.9, lon=32.8,
                            start='2025-04-01', end='2026-04-01'):
    """Fetch historical hourly weather from Open-Meteo Archive API."""
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'hourly': ['temperature_2m','cloudcover',
                   'windspeed_10m','shortwave_radiation'],
        'timezone': 'Europe/Istanbul'
    }
    r  = requests.get(url, params=params)
    df = pd.DataFrame(r.json()['hourly'])
    df['datetime'] = pd.to_datetime(df['time'])
    df['hour']     = df['datetime'].dt.hour
    df['month']    = df['datetime'].dt.month
    return df[['datetime','hour','month','temperature_2m',
               'cloudcover','windspeed_10m','shortwave_radiation']]

def get_weather_forecast(lat=39.9, lon=32.8, days=2):
    """Fetch hourly weather forecast from Open-Meteo Forecast API."""
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude': lat, 'longitude': lon,
        'hourly': ['temperature_2m','cloudcover',
                   'windspeed_10m','shortwave_radiation'],
        'forecast_days': days,
        'timezone': 'Europe/Istanbul'
    }
    r  = requests.get(url, params=params)
    df = pd.DataFrame(r.json()['hourly'])
    df['datetime'] = pd.to_datetime(df['time'])
    df['hour']     = df['datetime'].dt.hour
    df['month']    = df['datetime'].dt.month
    return df[['datetime','hour','month','temperature_2m',
               'cloudcover','windspeed_10m','shortwave_radiation']]

print('Fetching historical weather...')
df_weather = get_weather_historical()
print(f'Weather rows: {len(df_weather)}')

---
## 3 — Feature Engineering

Merge all data sources and compute derived features:
- `net_surplus` = renewable generation − consumption
- `re_share` = renewable / total generation × 100
- `price_lag24` = same hour yesterday
- `price_lag168` = same hour last week

In [ ]:
# Merge all datasets
df_tr['datetime']      = pd.to_datetime(df_tr['datetime'])
df_weather['datetime'] = pd.to_datetime(df_weather['datetime'])
rt_gen['datetime']     = pd.to_datetime(rt_gen['datetime'])
rt_cons['datetime']    = pd.to_datetime(rt_cons['datetime'])

# Step 1: PTF + weather
df = df_tr[['datetime','hour','month','price']].merge(
    df_weather[['datetime','temperature_2m','cloudcover',
                'windspeed_10m','shortwave_radiation']],
    on='datetime', how='inner'
)

# Step 2: + generation
df = df.merge(
    rt_gen[['datetime','total','renewable','wind','sun']],
    on='datetime', how='inner'
)

# Step 3: + consumption
df = df.merge(
    rt_cons[['datetime','consumption_mwh']],
    on='datetime', how='inner'
)

# Numeric conversion
for col in ['renewable','consumption_mwh','total','wind','sun']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Derived features
df['net_surplus'] = df['renewable'] - df['consumption_mwh']
df['re_share']    = df['renewable'] / df['total'] * 100

# Lag features
df = df.sort_values('datetime').reset_index(drop=True)
df['price_lag24']  = df['price'].shift(24)
df['price_lag168'] = df['price'].shift(168)
df['dayofweek']    = df['datetime'].dt.dayofweek
df['is_weekend']   = (df['dayofweek'] >= 5).astype(int)
df = df.dropna().reset_index(drop=True)

print(f'Final dataset: {len(df)} hours')
print(f'\nCorrelations with PTF:')
for col in ['re_share','net_surplus','renewable',
            'shortwave_radiation','wind','temperature_2m']:
    print(f'  {col:25s}: {df[col].corr(df["price"]):+.3f}')

---
## 4 — Model Training

### Model v1 — weather + lag only
### Model v2 — weather + generation + consumption + lag

**Features:**
- Time: hour, month, day of week, is_weekend
- Weather: temperature, cloud cover, wind speed, solar radiation
- Generation: renewable total, wind, solar
- Balance: consumption, net surplus, renewable share
- Lag prices: t-24h, t-168h

In [ ]:
FEATURES = [
    'hour', 'month', 'dayofweek', 'is_weekend',
    'temperature_2m', 'cloudcover',
    'windspeed_10m',  'shortwave_radiation',
    'renewable',      'wind', 'sun',
    'consumption_mwh','net_surplus', 're_share',
    'price_lag24',    'price_lag168'
]

X = df[FEATURES]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False)

rf = RandomForestRegressor(
    n_estimators=100, max_depth=12,
    random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f'PowerBid Model v2')
print(f'MAE : {mae:.1f} TRY/MWh')
print(f'R2  : {r2:.3f}')
print(f'Test: {len(y_test)} hours')

# Feature importance
importances = pd.Series(
    rf.feature_importances_, index=FEATURES
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
importances.plot(kind='bar', ax=ax, color='#378ADD', alpha=0.8)
ax.set_title('Feature importance — PowerBid Random Forest')
ax.set_ylabel('Importance score')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()
print(importances.round(3))

---
## 5 — Decision Engine

For each hour of the next day, PowerBid outputs:
- **PTF forecast** — predicted market clearing price
- **Confidence interval** — ±0.5 std of historical prices for that hour
- **Recommended bid** — conservative bid at 85% of forecast
- **Acceptance probability** — % of historical hours where PTF ≥ bid

In [ ]:
# Fetch tomorrow's weather forecast
df_forecast = get_weather_forecast(days=2)
tomorrow    = (pd.Timestamp.now() + pd.Timedelta(days=1)).date()
df_tomorrow = df_forecast[
    df_forecast['datetime'].dt.date == tomorrow
].copy().reset_index(drop=True)

print(f"Tomorrow's forecast ({tomorrow}): {len(df_tomorrow)} hours")
print(df_tomorrow[['datetime','hour','temperature_2m',
                   'cloudcover','shortwave_radiation']].to_string(index=False))

In [ ]:
# Decision engine — generate bid recommendations for each hour
results = []

for _, row in df_tomorrow.iterrows():
    h = int(row['hour'])

    # Lag prices from most recent data
    h_data   = df[df['hour'] == h]['price']
    lag24    = h_data.iloc[-1]  if len(h_data) >= 1 else df['price'].mean()
    lag168   = h_data.iloc[-7]  if len(h_data) >= 7 else lag24

    # Estimate generation features from weather
    solar_est    = row['shortwave_radiation'] * 0.05
    re_est       = 15000 + solar_est * 10
    cons_est     = 35000 - row['temperature_2m'] * 200
    cons_est     = max(cons_est, 20000)
    re_share_est = min(re_est / 45000 * 100, 80)

    X_pred = pd.DataFrame([{
        'hour'               : h,
        'month'              : int(row['month']),
        'dayofweek'          : pd.Timestamp(row['datetime']).dayofweek,
        'is_weekend'         : int(pd.Timestamp(row['datetime']).dayofweek >= 5),
        'temperature_2m'     : row['temperature_2m'],
        'cloudcover'         : row['cloudcover'],
        'windspeed_10m'      : row['windspeed_10m'],
        'shortwave_radiation': row['shortwave_radiation'],
        'renewable'          : re_est,
        'wind'               : 5000,
        'sun'                : solar_est,
        'consumption_mwh'    : cons_est,
        'net_surplus'        : re_est - cons_est,
        're_share'           : re_share_est,
        'price_lag24'        : lag24,
        'price_lag168'       : lag168,
    }])

    ptf_pred = rf.predict(X_pred)[0]
    h_std    = h_data.std()
    q25      = h_data.quantile(0.25)
    bid      = min(ptf_pred * 0.85, q25)
    accept_p = (h_data >= bid).mean() * 100

    results.append({
        'hour'          : h,
        'ptf_forecast'  : round(ptf_pred),
        'lower_bound'   : round(max(0, ptf_pred - h_std * 0.5)),
        'upper_bound'   : round(ptf_pred + h_std * 0.5),
        'recommended_bid': round(bid),
        'acceptance_pct': round(accept_p, 1),
        'risk'          : 'LOW' if accept_p >= 85 else
                          'MED' if accept_p >= 75 else 'HIGH'
    })

df_report = pd.DataFrame(results)
print(f'PowerBid Bid Report — {tomorrow}')
print(f'{"Hour":>5} {"Forecast":>10} {"Lower":>8} {"Upper":>8} '
      f'{"Bid":>10} {"Accept%":>9} {"Risk":>6}')
print('-' * 65)
for _, r in df_report.iterrows():
    print(f"{r['hour']:>5} {r['ptf_forecast']:>10,.0f} "
          f"{r['lower_bound']:>8,.0f} {r['upper_bound']:>8,.0f} "
          f"{r['recommended_bid']:>10,.0f} {r['acceptance_pct']:>8.1f}% "
          f"{r['risk']:>6}")

---
## 6 — Daily Bid Report Visualisation

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

hours = df_report['hour']

# Plot 1: PTF forecast + confidence interval + bid
axes[0].fill_between(hours,
                     df_report['lower_bound'],
                     df_report['upper_bound'],
                     alpha=0.25, color='#378ADD', label='Confidence interval')
axes[0].plot(hours, df_report['ptf_forecast'],
             color='#378ADD', linewidth=2.5,
             marker='o', markersize=5, label='PTF forecast')
axes[0].plot(hours, df_report['recommended_bid'],
             color='#1D9E75', linewidth=2, marker='s', markersize=4,
             linestyle='--', label='Recommended bid')
axes[0].set_ylabel('TRY/MWh')
axes[0].set_title(f'PowerBid — PTF forecast and bid recommendation')
axes[0].legend(fontsize=9)
axes[0].set_xticks(range(24))

# Plot 2: acceptance probability
bar_colors = ['#1D9E75' if p >= 85 else
              '#EF9F27' if p >= 75 else
              '#E24B4A' for p in df_report['acceptance_pct']]
axes[1].bar(hours, df_report['acceptance_pct'],
            color=bar_colors, alpha=0.85)
axes[1].axhline(85, color='#1D9E75', linewidth=1.5,
                linestyle='--', label='85% threshold (LOW risk)')
axes[1].axhline(75, color='#EF9F27', linewidth=1.5,
                linestyle='--', label='75% threshold (MED risk)')
axes[1].set_ylabel('Acceptance probability (%)')
axes[1].set_xlabel('Hour of day')
axes[1].set_title('Bid acceptance probability — green>85%, amber>75%, red<75%')
axes[1].legend(fontsize=9)
axes[1].set_xticks(range(24))
axes[1].set_ylim(0, 105)

plt.suptitle(f'PowerBid Daily Bid Report — {tomorrow}',
             fontsize=14, fontweight='500', y=1.01)
plt.tight_layout()
plt.show()

# Summary
low  = (df_report['risk'] == 'LOW').sum()
med  = (df_report['risk'] == 'MED').sum()
high = (df_report['risk'] == 'HIGH').sum()
print(f'Risk summary: LOW={low}h  MED={med}h  HIGH={high}h')
print(f'Best bid hours (LOW risk): '
      f'{df_report[df_report["risk"]=="LOW"]["hour"].tolist()}')